# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4: CTR / Engagement Opportunity Scoring.**

I am picking Lane 4 because it follows directly from what I already found in the Week 1 starter
notebooks. In notebook 01 I looked at click through rate while holding search position fixed, and
I saw that pages which sit at the same position tier can still earn very different CTR, and that
some content types (comparison articles in that sample) consistently under convert their
impressions into clicks. That observation is exactly the question Lane 4 asks: *which visible pages
under capture clicks relative to what their position would predict, and therefore deserve a title,
meta, or snippet review?*

It fits the data I have. The starter dataset ships the observable signals this lane needs
(impressions, clicks, CTR, average position, position tier, content type, age, sessions and
engagement), and the larger warehouse release can extend the same idea later. It is also a genuine
decision support problem rather than a "train a model for its own sake" exercise: a review team has
limited time, and the useful output is a ranked shortlist a human can act on. I can confirm or
change this lane until the end of Week 4, but the numbers below already make it look worth the next
seven weeks.

In [1]:
# Setup: locate the repo root so this runs both locally and on Colab, then load the starter data.
import os, sys, subprocess
import pandas as pd, numpy as np

if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

CSV = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV), "starter CSV not found -- are you at the repo root?"
df = pd.read_csv(CSV)

# Grain check: one row per content page.
print("Grain: one row per content page")
print(f"  pages (rows)      : {len(df):,}")
print(f"  unique content_id : {df['content_id'].nunique():,}")
print(f"  clients           : {df['client_id'].nunique()}")

# How big is the population this lane serves? Only pages that are actually visible can
# 'under capture' clicks, so I size the visible and well positioned set.
visible = df[df["impressions_90d"] >= 500]
positioned = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)]
print(f"\nAddressable population for a CTR review lane:")
print(f"  visible pages (impressions_90d >= 500)          : {len(visible):,}  ({len(visible)/len(df)*100:.1f}% of pages)")
print(f"  visible AND ranking in positions 1-20           : {len(positioned):,}  ({len(positioned)/len(df)*100:.1f}% of pages)")

Grain: one row per content page
  pages (rows)      : 30,000
  unique content_id : 30,000
  clients           : 32

Addressable population for a CTR review lane:
  visible pages (impressions_90d >= 500)          : 16,726  (55.8% of pages)
  visible AND ranking in positions 1-20           : 12,023  (40.1% of pages)


## 2. The question: decision, action, cost of a wrong call

**Search question.** Among pages that are already visible in search, which ones under capture clicks
relative to other pages at the same search position, and therefore deserve a title, meta, or snippet
review first?

**Unit of analysis (grain).** One content page (one row per `content_id` in the starter slice).

**Output.** A ranked review queue: pages ordered by how far their CTR sits below what their position
tier would predict, each carrying a short reason code (for example *high impressions, strong
position, low CTR*) and a confidence label.

**The decision it improves.** A content or SEO reviewer has limited time and cannot inspect every
page. Today that triage is done by rules of thumb. This work improves *which pages the reviewer opens
first*, so scarce review capacity lands on the pages with the most visible upside.

**Who acts, and how.** A human reviewer takes the queue and, page by page, decides to rewrite the
title or meta description, improve the snippet or intent match, or simply monitor. The system
recommends; the person decides.

**Cost of a wrong recommendation.** A false positive spends reviewer time on a page whose low CTR is
actually noise, a genuinely low interest query, or a SERP feature effect, and in the worst case a
reviewer edits a page that was fine and makes it worse. A false negative leaves a real opportunity
unfixed. Because the action costs human time and carries edit risk, the ranking has to be honest
about confidence, not just confident.

**Why data or ML helps at all.** Expected CTR depends heavily on position, so a flat "low CTR" rule
is misleading: a page at position 9 with low CTR may be perfectly normal, while a page at position 2
with the same CTR is a real problem. Adjusting CTR for position (a residual or gap approach) and
ranking by that gap is something you cannot eyeball across 30,000 pages, and it is exactly where a
simple, explainable model or scoring method beats a fixed threshold.

In [2]:
# Why this needs position adjustment (not just "sort by low CTR"):
# CTR is strongly tied to position, so a fair comparison must hold position fixed.
d = df[(df["impressions_90d"] >= 100) & (df["ctr"].notna())].copy()
tier_ctr = d.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Mean CTR by position tier (pages with impressions_90d >= 100):")
print(tier_ctr.round(4).to_string())
print("\nCTR falls steeply as position worsens, so 'low CTR' only means something")
print("relative to a page's own position tier -- that is what makes this an analysis problem,")
print("not a one-line sort.")

Mean CTR by position tier (pages with impressions_90d >= 100):
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554

CTR falls steeply as position worsens, so 'low CTR' only means something
relative to a page's own position tier -- that is what makes this an analysis problem,
not a one-line sort.


## 3. Quick look at the data (2-3 real numbers)

Below I size the actual opportunity. For every visible page I compare its CTR to the **median CTR of
its own position tier**, so I am comparing like for like. Two headline numbers come out: how many
visible pages sit below their tier's typical CTR, and how many sit at less than half of it (a clear,
position adjusted opportunity pool).

In [3]:
# Position-adjusted CTR gap: compare each page to the MEDIAN CTR of its own position tier.
d = df[(df["impressions_90d"] >= 100) & (df["ctr"].notna())].copy()
d["tier_median_ctr"] = d.groupby("position_tier")["ctr"].transform("median")
d["ctr_gap"] = d["ctr"] - d["tier_median_ctr"]

n = len(d)
below      = (d["ctr_gap"] < 0).sum()
far_below  = (d["ctr"] < 0.5 * d["tier_median_ctr"]).sum()

print(f"Visible pages examined (impressions_90d >= 100): {n:,}")
print(f"  below their own position tier's median CTR          : {below:,}  ({below/n*100:.1f}%)")
print(f"  at LESS THAN HALF their tier's median CTR (clear gap): {far_below:,}  ({far_below/n*100:.1f}%)")

# A concrete slice a reviewer could act on: strong position, real volume, weak CTR for that position.
review_pool = d[(d["avg_position"] > 0) & (d["avg_position"] <= 20)
                & (d["impressions_90d"] >= 500)
                & (d["ctr"] < 0.5 * d["tier_median_ctr"])]
print(f"\nExample high-value review pool (position 1-20, impressions_90d >= 500,")
print(f"CTR below half the tier median): {len(review_pool):,} pages")
print("\nObserved / directional: roughly a third of visible pages under capture clicks relative to")
print("peers at the same position. That is a large, rankable opportunity pool -- enough to justify")
print("building an honest, position-adjusted review queue over the next seven weeks.")

Visible pages examined (impressions_90d >= 100): 22,006
  below their own position tier's median CTR          : 10,307  (46.8%)
  at LESS THAN HALF their tier's median CTR (clear gap): 7,239  (32.9%)

Example high-value review pool (position 1-20, impressions_90d >= 500,
CTR below half the tier median): 2,932 pages

Observed / directional: roughly a third of visible pages under capture clicks relative to
peers at the same position. That is a large, rankable opportunity pool -- enough to justify
building an honest, position-adjusted review queue over the next seven weeks.


## 4. Careful words: what I can and can't claim

**What this work will be able to say.** It will produce *observed, position adjusted* comparisons:
that a page earns a lower CTR than other pages at the same search position, measured on this
snapshot. It is *decision support* — it ranks candidates so a human reviews the most promising first.
Every claim will use careful language ("we observed", "this suggests", "directional"), report the
thresholds I chose, and pass a leakage check so no feature secretly encodes the outcome.

**What it will never claim.** It will not claim that rewriting a title *causes* more clicks — proving
cause needs an experiment or a causal design this data alone cannot provide. It will not claim to
predict or reveal Google's ranking algorithm, or to measure AI citations or rankings. A low CTR gap
has benign explanations too (brand versus non brand queries, SERP features, intent mismatch,
low volume noise), so the queue is a prompt for human review, not a verdict. And nothing public
facing will contain client names, domains, URLs, or raw queries — only pseudonymized IDs and
aggregated, position adjusted metrics.

In [4]:
# Public-safety and leakage sanity check on the fields this lane uses.
lane_fields = ["content_id", "client_id", "impressions_90d", "clicks_90d",
               "ctr", "avg_position", "position_tier", "content_type",
               "content_age_days", "sessions_90d", "engagement_rate"]
present = [c for c in lane_fields if c in df.columns]
print("Lane fields available (all observable signals, no product decision flags):")
print("  " + ", ".join(present))

# Confirm no raw, re-identifying text columns are present in the release.
risky = [c for c in df.columns if any(k in c.lower() for k in ["url", "query", "keyword", "title", "domain", "name"])]
print(f"\nRaw re-identifying columns present: {risky if risky else 'none'}")
print("IDs are pseudonymized; the label idea (future CTR gap) is defined by me, not copied from a")
print("product score -- so there is no obvious leakage or private-data issue at the framing stage.")

Lane fields available (all observable signals, no product decision flags):
  content_id, client_id, impressions_90d, clicks_90d, ctr, avg_position, position_tier, content_type, content_age_days, sessions_90d, engagement_rate

Raw re-identifying columns present: none
IDs are pseudonymized; the label idea (future CTR gap) is defined by me, not copied from a
product score -- so there is no obvious leakage or private-data issue at the framing stage.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.